In [6]:
# ==============================
# IMPORTS
# ==============================

import os
import requests
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import geodatasets
import cv2
import matplotlib.pyplot as plt
import geopandas as gpd

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from shapely.geometry import Point

# ==============================
# CONFIG
# ==============================

MAPILLARY_TOKEN = "YOUR_TOKEN"

DATA_DIR = "data"
IMAGE_DIR = f"{DATA_DIR}/images"
META_PATH = f"{DATA_DIR}/metadata.csv"

NUM_BBOXES = 1500
IMAGES_PER_BBOX = 8
K_REGIONS = 500

BATCH_SIZE = 64
EPOCHS = 1000
ALPHA = 1.0
BETA = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(IMAGE_DIR, exist_ok=True)

c:\Users\Tyrese\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def random_bbox():
    land_regions = [
        (30, 50, -10, 30),
        (25, 48, -125, -70),
        (-35, -15, 115, 150),
        (10, 45, 60, 140)
    ]

    lat_min, lat_max, lon_min, lon_max = land_regions[np.random.randint(len(land_regions))]
    lat = np.random.uniform(lat_min, lat_max)
    lon = np.random.uniform(lon_min, lon_max)

    d = 0.01
    return [lon-d, lat-d, lon+d, lat+d]


class CountryLookup:

    def __init__(self):
        try:
            path = geodatasets.get_path("naturalearth_lowres")
            self.world = gpd.read_file(path)
        except:
            url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
            self.world = gpd.read_file(url)

        if 'name' not in self.world.columns and 'ADMIN' in self.world.columns:
            self.world = self.world.rename(columns={'ADMIN': 'name'})

        self.world = self.world[["name", "geometry"]]

    def lookup(self, lat, lon):

        point = Point(lon, lat)
        match = self.world[self.world.contains(point)]

        return match.iloc[0]["name"] if not match.empty else "Unknown"


class MapillaryDownloader:

    def __init__(self, token, country_lookup):

        self.token = token
        self.base_url = "https://graph.mapillary.com/images"
        self.headers = {"Authorization": f"OAuth {token}"}
        self.country_lookup = country_lookup

    def fetch_metadata(self, bbox):

        params = {
            "fields": "id,thumb_1024_url,geometry",
            "bbox": ",".join(map(str, bbox)),
            "limit": IMAGES_PER_BBOX
        }

        try:
            r = requests.get(self.base_url, params=params, headers=self.headers, timeout=10)
            return r.json().get("data", []) if r.status_code == 200 else []
        except:
            return []

    def download_dataset(self):

        records = []

        while len(records) < 100:

            bbox = random_bbox()
            metadata = self.fetch_metadata(bbox)

            for img in metadata:

                url = img.get("thumb_1024_url")
                if not url:
                    continue

                try:

                    r = requests.get(url, timeout=10)

                    if r.status_code != 200:
                        continue

                    path = f"{IMAGE_DIR}/{img['id']}.jpg"

                    with open(path, "wb") as f:
                        f.write(r.content)

                    lon, lat = img["geometry"]["coordinates"]

                    records.append({
                        "path": path,
                        "lat": lat,
                        "lon": lon,
                        "country": self.country_lookup.lookup(lat, lon)
                    })

                except:
                    continue

        return pd.DataFrame(records)

In [ ]:
if os.path.exists(META_PATH):

    print("Loading cached dataset...")
    df = pd.read_csv(META_PATH)

else:

    print("Downloading dataset...")
    cl = CountryLookup()
    dl = MapillaryDownloader(MAPILLARY_TOKEN, cl)

    df = dl.download_dataset()

    df.to_csv(META_PATH, index=False)

print("Dataset size:", len(df))

In [7]:
 df = pd.read_csv(META_PATH)

FileNotFoundError: [Errno 2] No such file or directory: 'data/metadata.csv'

In [2]:
def create_regions(df):

    coords = df[["lat", "lon"]].values
    k = min(K_REGIONS, len(coords))

    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)

    df["region"] = kmeans.fit_predict(coords)

    return df, kmeans.cluster_centers_


def spatial_split(df):

    df["tile"] = df.lat.astype(int).astype(str) + "_" + df.lon.astype(int).astype(str)

    tiles = df["tile"].unique()

    tr_t, te_t = train_test_split(tiles, test_size=0.2, random_state=42)
    tr_t, va_t = train_test_split(tr_t, test_size=0.2, random_state=42)

    return (
        df[df.tile.isin(tr_t)],
        df[df.tile.isin(va_t)],
        df[df.tile.isin(te_t)]
    )


df, centroids = create_regions(df)

train_df, val_df, test_df = spatial_split(df)

NameError: name 'df' is not defined

In [ ]:
class GeoDataset(Dataset):

    def __init__(self, df, country_map, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.country_map = country_map

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = Image.open(row.path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        c = torch.tensor(self.country_map.get(row.country, 0))
        r = torch.tensor(row.region)

        return img, c, r, torch.tensor(row.lat), torch.tensor(row.lon)


class GeoModel(nn.Module):

    def __init__(self, num_countries, num_regions):

        super().__init__()

        res = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        self.backbone = nn.Sequential(*list(res.children())[:-2])

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.c_head = nn.Linear(2048, num_countries)
        self.r_head = nn.Linear(2048, num_regions)

    def forward(self, x):

        x = self.pool(self.backbone(x))
        x = torch.flatten(x, 1)

        return self.c_head(x), self.r_head(x)

In [1]:
c_map = {c: i for i, c in enumerate(sorted(df.country.unique()))}

tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

tr_ld = DataLoader(GeoDataset(train_df, c_map, tf), batch_size=BATCH_SIZE, shuffle=True)
te_ld = DataLoader(GeoDataset(test_df, c_map, tf), batch_size=BATCH_SIZE)

model = GeoModel(len(c_map), len(centroids)).to(device)

opt = optim.Adam(model.parameters(), lr=1e-4)

crit = nn.CrossEntropyLoss()

for ep in range(EPOCHS):

    model.train()
    total_loss = 0

    for imgs, c_t, r_t, _, _ in tr_ld:

        imgs, c_t, r_t = imgs.to(device), c_t.to(device), r_t.to(device)

        opt.zero_grad()

        c_l, r_l = model(imgs)

        loss = ALPHA*crit(c_l, c_t) + BETA*crit(r_l, r_t)

        loss.backward()
        opt.step()

        total_loss += loss.item()

    print("Epoch", ep, "Loss", total_loss/len(tr_ld))

NameError: name 'df' is not defined

sdfsd

In [ ]:
import os
import requests
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import geopandas as gpd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from shapely.geometry import Point

# ==============================
# CONFIG & CONSTANTS
# ==============================
MAPILLARY_TOKEN = "MLY|26664131136518032|9af8051ce63bc8d3adb2abc9b889ed15"
DATA_DIR = "data"
IMAGE_DIR = f"{DATA_DIR}/images"
META_PATH = f"{DATA_DIR}/metadata.csv"

K_REGIONS = 50       
MAX_IMAGES_PER_REGION = 200 
EARTH_RADIUS_KM = 6371.0
BATCH_SIZE = 32
EPOCHS = 10 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(IMAGE_DIR, exist_ok=True)

# ==============================
# GEOGRAPHIC UTILITIES
# ==============================
def latlon_to_cartesian(lat, lon):
    lat_rad, lon_rad = np.radians(lat), np.radians(lon)
    return np.cos(lat_rad)*np.cos(lon_rad), np.cos(lat_rad)*np.sin(lon_rad), np.sin(lat_rad)

def cartesian_to_latlon(x, y, z):
    return np.degrees(np.arcsin(z)), np.degrees(np.arctan2(y, x))

def haversine_dist(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(torch.deg2rad, [lat1, lon1, lat2, lon2])
    a = torch.sin((lat2-lat1)/2)**2 + torch.cos(lat1)*torch.cos(lat2)*torch.sin((lon2-lon1)/2)**2
    return 2 * EARTH_RADIUS_KM * torch.arcsin(torch.sqrt(a))

# ==============================
# DATA ACQUISITION (MAPILLARY)
# ==============================
def download_data(num_images=200):
    print(f"Data missing. Downloading {num_images} images from Mapillary...")
    
    # Robust World Map Loading
    try:
        world_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
        world = gpd.read_file(world_url)
    except:
        # Emergency backup if external URL fails
        path = gpd.datasets.get_path('naturalearth_lowres')
        world = gpd.read_file(path)

    records = []
    while len(records) < num_images:
        lat, lon = np.random.uniform(-60, 70), np.random.uniform(-180, 180)
        bbox = f"{lon-0.1},{lat-0.1},{lon+0.1},{lat+0.1}"
        url = f"https://graph.mapillary.com/images?fields=id,thumb_1024_url,geometry&bbox={bbox}&limit=5"
        
        try:
            res = requests.get(url, headers={"Authorization": f"OAuth {MAPILLARY_TOKEN}"}, timeout=10).json()
            for img in res.get('data', []):
                if 'thumb_1024_url' not in img: continue
                img_id = img['id']
                path = f"{IMAGE_DIR}/{img_id}.jpg"
                
                # Image Download
                img_res = requests.get(img['thumb_1024_url'], timeout=10)
                if img_res.status_code == 200:
                    with open(path, "wb") as f: f.write(img_res.content)
                else: continue
                
                lon_val, lat_val = img['geometry']['coordinates']
                pt = Point(lon_val, lat_val)
                match = world[world.contains(pt)]
                country = match.iloc[0]['ADMIN'] if not match.empty else "Unknown"
                
                records.append({"path": path, "lat": lat_val, "lon": lon_val, "country": country})
                if len(records) >= num_images: break
            print(f"Downloaded: {len(records)}/{num_images}", end="\r")
        except: continue
            
    df = pd.DataFrame(records)
    df.to_csv(META_PATH, index=False)
    print("\nDownload Complete.")
    return df

# ==============================
# MODEL & TRAINING
# ==============================
class GeoDataset(Dataset):
    def __init__(self, df, c_map, transform=None):
        self.df, self.c_map, self.transform = df, c_map, transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = self.transform(Image.open(row.path).convert("RGB"))
        return img, torch.tensor(self.c_map[row.country], dtype=torch.long), \
               torch.tensor(row.region, dtype=torch.long), \
               torch.tensor(row.lat, dtype=torch.float32), torch.tensor(row.lon, dtype=torch.float32)

class ResNetGeo(nn.Module):
    def __init__(self, n_c, n_r):
        super().__init__()
        # Using ResNet-50 as per section 4 of your proposal
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.c_head = nn.Linear(2048, n_c)
        self.r_head = nn.Linear(2048 + n_c, n_r)
    def forward(self, x):
        f = torch.flatten(self.features(x), 1)
        c = self.c_head(f)
        r = self.r_head(torch.cat((f, c), dim=1))
        return c, r

# ==============================
# EXECUTION
# ==============================
df = pd.read_csv(META_PATH) if os.path.exists(META_PATH) else download_data(200)



# 3D Cartesian Clustering
x, y, z = latlon_to_cartesian(df['lat'], df['lon'])
kmeans = KMeans(n_clusters=min(K_REGIONS, len(df)), n_init=10).fit(np.stack([x, y, z], 1))
df['region'] = kmeans.labels_
centroids = torch.tensor([cartesian_to_latlon(*c) for c in kmeans.cluster_centers_], dtype=torch.float32).to(device)

c_map = {c: i for i, c in enumerate(df['country'].unique())}
train_df, val_df = train_test_split(df, test_size=0.2)

tf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), 
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

tr_ld = DataLoader(GeoDataset(train_df, c_map, tf), batch_size=BATCH_SIZE, shuffle=True)
va_ld = DataLoader(GeoDataset(val_df, c_map, tf), batch_size=BATCH_SIZE)

model = ResNetGeo(len(c_map), len(centroids)).to(device)
opt = optim.Adam(model.parameters(), lr=1e-4)
crit = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    for imgs, c_t, r_t, _, _ in tr_ld:
        opt.zero_grad()
        c_p, r_p = model(imgs.to(device))
        loss = crit(c_p, c_t.to(device)) + crit(r_p, r_t.to(device))
        loss.backward(); opt.step()
    
    model.eval()
    err = 0
    with torch.no_grad():
        for imgs, _, _, t_lat, t_lon in va_ld:
            _, r_p = model(imgs.to(device))
            p_idx = r_p.argmax(1)
            p_lat, p_lon = centroids[p_idx, 0], centroids[p_idx, 1]
            err += haversine_dist(t_lat.to(device), t_lon.to(device), p_lat, p_lon).sum().item()
    
    
    print(f"Epoch {epoch+1} | Mean Error: {err/len(val_df):.2f} km")

Data missing. Downloading 200 images from Mapillary...


Metadata not found. You must implement and run the MapillaryDownloader first.


FileNotFoundError: [Errno 2] No such file or directory: 'data/metadata.csv'

: 